# ColdStart Killer - Buyer Search Demo

This notebook demonstrates the end-to-end buyer search path from raw query to ranked, explainable results. The flow is Query Processing -> Hybrid Search -> Results -> Cold Start Analysis -> Debug Summary. Run the startup check first so Python 3.14 and heavy dependency issues show up clearly.


In [1]:
# ============================================
# STARTUP CHECK - Run this cell first
# ============================================
import sys
import subprocess

print(f"Python version: {sys.version}")
print(f"Platform: {sys.platform}")

# Check critical packages in the current kernel.
checks = {}
for pkg in ["pymongo", "pydantic", "numpy", "dotenv"]:
    try:
        __import__(pkg.replace("-", "_"))
        checks[pkg] = "\u2705 OK"
    except ImportError:
        checks[pkg] = "\u274c Missing"
    except Exception as e:
        checks[pkg] = f"\u26a0\ufe0f Error: {e}"

# Check heavy packages in a child process so a native crash does not kill this kernel.
for pkg in ["torch", "sentence_transformers"]:
    try:
        result = subprocess.run(
            [sys.executable, "-c", f"import {pkg}; print('ok')"],
            capture_output=True,
            text=True,
            timeout=45,
        )
        if result.returncode == 0:
            checks[pkg] = "\u2705 OK"
        else:
            error = (result.stderr or result.stdout or "import failed").strip().splitlines()[-1]
            checks[pkg] = f"\u26a0\ufe0f Error: {error}"
    except subprocess.TimeoutExpired:
        checks[pkg] = "\u26a0\ufe0f Error: import timed out"
    except Exception as e:
        checks[pkg] = f"\u26a0\ufe0f Error: {e}"

print("\nPackage check:")
for pkg, status in checks.items():
    print(f"  {pkg}: {status}")

# Check .env
from pathlib import Path
env_exists = (Path.cwd() / ".env").exists() or \
             (Path.cwd().parent / ".env").exists()
print(f"\n.env file: {'\u2705 Found' if env_exists else '\u274c Missing'}")


Python version: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
Platform: win32

Package check:
  pymongo: ✅ OK
  pydantic: ✅ OK
  numpy: ✅ OK
  dotenv: ✅ OK
  torch: ✅ OK
  sentence_transformers: ✅ OK

.env file: ✅ Found


## ⚙️ Bước 0: Cấu hình — Nhập query tại đây

Đây là điểm bắt đầu. Thay đổi `QUERY` để test các loại sản phẩm khác nhau. `TOP_K` kiểm soát số kết quả trả về trong demo.


In [2]:
# ============================================================
# ⚙️ CONFIGURATION — Thay đổi query và số kết quả ở đây
# ============================================================

# ← ĐỔI QUERY Ở ĐÂY để test các loại query khác nhau
QUERY: str = "ốp điện thoại samsung galaxy a14 dưới 500k"
TOP_K: int = 10

# Ví dụ các query để test:
# QUERY = "moisturizing cream for dry skin"
# QUERY = "phone case samsung galaxy s22 under 300k"
# QUERY = "sạc nhanh samsung galaxy a14 dưới 500k"
# QUERY = "wireless charger iphone 14"
# QUERY = "váy đầm dự tiệc đẹp"
# QUERY = "tai nghe chống ồn dưới 500k"  # may return weak matches if headphones are sparse in the dataset

print(f"Query: {QUERY!r}")
print(f"Top-K: {TOP_K}")


Query: 'ốp điện thoại samsung galaxy a14 dưới 500k'
Top-K: 10


## Cell A - Safe Imports

This cell loads lightweight packages and sets the project root. It gives the notebook a stable import base before MongoDB or embedding dependencies are touched. Expected output is the project root path.


In [3]:
# Cell A - Safe imports only
import json
import os
import sys
from pathlib import Path

import pymongo

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    get_ipython().run_line_magic("cd", str(ROOT.parent))
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Safe imports loaded.")
print("Project root:", ROOT)


c:\HCMUS\MONGODB\coldstart+project\ColdStart_Killer
Safe imports loaded.
Project root: c:\HCMUS\MONGODB\coldstart+project\ColdStart_Killer


## Cell B - MongoDB Imports and Counts

This cell imports MongoDB helpers safely, pings Atlas, and prints collection counts. Watch for connected status and non-zero retrieval units. This confirms the cold-start index data is present.


In [4]:
# Cell B - MongoDB imports and connection check
try:
    from src.mongodb import get_items_collection, get_retrieval_units_collection, ping_mongodb
    MONGODB_AVAILABLE = True
except ImportError as e:
    MONGODB_AVAILABLE = False
    print(f"WARNING: MongoDB helpers not available: {e}")
except Exception as e:
    MONGODB_AVAILABLE = False
    print(f"WARNING: Unexpected error loading MongoDB helpers: {e}")

if not MONGODB_AVAILABLE:
    raise RuntimeError("src.mongodb is required. Check imports above.")

items = get_items_collection()
retrieval_units = get_retrieval_units_collection()

ping = ping_mongodb()
print("MongoDB connection:", "PASS" if ping.get("ok") else "FAIL")
if not ping.get("ok"):
    raise RuntimeError(ping.get("error", "MongoDB ping failed"))

collection_counts = {
    "total_items": items.count_documents({}),
    "total_retrieval_units": retrieval_units.count_documents({}),
    "hype_question_units_with_embedding": retrieval_units.count_documents({
        "unit_type": "hype_question",
        "embedding": {"$exists": True},
    }),
    "proposition_units_with_text_search": retrieval_units.count_documents({
        "unit_type": "proposition",
        "text_search": {"$exists": True},
    }),
    "cold_start_items": items.count_documents({"cold_start.is_cold_item": True}),
}

print("\nCollection health")
print("Metric | Count")
print("--- | ---:")
for metric, count in collection_counts.items():
    print(f"{metric} | {count}")

collection_health_pass = all(count > 0 for count in collection_counts.values())
print("\nCollection health status:", "PASS" if collection_health_pass else "FAIL")


MongoDB connection: PASS

Collection health
Metric | Count
--- | ---:
total_items | 3000
total_retrieval_units | 29753
hype_question_units_with_embedding | 13580
proposition_units_with_text_search | 16173
cold_start_items | 3000

Collection health status: PASS


## Cell C - Search Pipeline Imports

This cell loads the search aggregation code separately from the embedding stack. It should not touch torch, CUDA, or BGE-M3. Expected output confirms the search pipeline is ready.


In [5]:
# Cell C - Search pipeline imports, no torch dependency expected
try:
    from src.search_pipeline import run_search
    from src.retrieval_output import build_explainable_result
    SEARCH_PIPELINE_AVAILABLE = True
except ImportError as e:
    SEARCH_PIPELINE_AVAILABLE = False
    print(f"WARNING: search_pipeline not available: {e}")
except Exception as e:
    SEARCH_PIPELINE_AVAILABLE = False
    print(f"WARNING: Unexpected error loading search_pipeline: {e}")

if SEARCH_PIPELINE_AVAILABLE:
    print("Search pipeline imports loaded.")


Search pipeline imports loaded.


## Cell D - Heavy Query Processor Boundary

This cell imports query processing with a visible BGE-M3 warning. The model is loaded later when the query is processed, so import errors and embedding errors are easier to separate. Expected output is a clear availability status.


In [6]:
# Cell D - Heavy query processor import boundary
print("WARNING: Loading BGE-M3 model (~1.5GB) may take a few minutes when process_query() runs.")
print("Importing query_processor should be lightweight; torch/sentence-transformers load during embedding.")

try:
    from src.query_processor import process_query
    QUERY_PROCESSOR_AVAILABLE = True
except ImportError as e:
    QUERY_PROCESSOR_AVAILABLE = False
    print(f"WARNING: query_processor not available: {e}")
except Exception as e:
    QUERY_PROCESSOR_AVAILABLE = False
    print(f"WARNING: Unexpected error loading query_processor: {e}")

if QUERY_PROCESSOR_AVAILABLE:
    print("query_processor import loaded. Embedding model will load on first process_query() call.")


Importing query_processor should be lightweight; torch/sentence-transformers load during embedding.
query_processor import loaded. Embedding model will load on first process_query() call.


## Step 1 - Query Processing

This step turns raw text into the search-ready fixture: language, translation, hard filters, HyPE text, BM25 text, and BGE-M3 embedding. Look at the embedding dimension and first values to confirm embedding ran. This connects the user query to the same semantic space as indexed HyPE units.


In [7]:
# Step 1: process query - detect language, translate, extract filters, embed
# process_query() returns a dict that can go directly into run_search().
if "QUERY" not in dir() or not isinstance(QUERY, str) or not QUERY.strip():
    raise RuntimeError("QUERY chưa được cấu hình. Hãy chạy cell Configuration trước.")
if "TOP_K" not in dir() or not isinstance(TOP_K, int) or TOP_K <= 0:
    raise RuntimeError("TOP_K chưa hợp lệ. Hãy chạy cell Configuration trước.")
if "QUERY_PROCESSOR_AVAILABLE" not in dir() or not QUERY_PROCESSOR_AVAILABLE:
    raise RuntimeError("query_processor is required. Check imports above.")

try:
    fixture = process_query(QUERY)
except Exception as e:
    raise RuntimeError(f"process_query failed. Check BGE-M3/torch/Ollama dependencies: {e}") from e

# Print intermediate outputs so the demo shows what the pipeline is doing.
print("Language detected:", fixture["language_detected"])
print("English translation:", fixture["english_query"])
print("Hard filters extracted:", json.dumps(fixture["hard_filters"], ensure_ascii=False))
print("HyPE query built:", fixture["hype_search_query_en"])
print("BM25 query built:", fixture["bm25_search_query_en"])

embedding = fixture["query_embedding"]
print("Embedding dimension:", len(embedding))
print("Embedding first 3 values:", embedding[:3])


c:\HCMUS\MONGODB\coldstart+project\ColdStart_Killer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.91it/s]

Language detected: vi
English translation: samsung galaxy a14 phone case under 500k
Hard filters extracted: {"in_stock": true, "price_max": 500000}
HyPE query built: user looking for samsung galaxy a14 phone case under 500k for everyday use
BM25 query built: samsung galaxy a14 phone case under 500k
Embedding dimension: 1024
Embedding first 3 values: [-0.027282696217298508, 0.010519112460315228, -0.06413625180721283]


## Step 2 - Hybrid Search

This step runs MongoDB hybrid retrieval with vector search and BM25, using the `unionWith` fallback. Look at the mode and result count. This is where cold-start items can rank using content and intent rather than interaction history.


In [8]:
# Step 2: run hybrid search - $vectorSearch + $search BM25 + RRF fusion
# mode="unionWith" uses the stable fallback and does not depend on native $rankFusion preview.
if "fixture" not in dir() or fixture is None:
    raise RuntimeError("fixture chưa được tạo. Hãy chạy cell Query Processing trước.")
if "TOP_K" not in dir() or not isinstance(TOP_K, int) or TOP_K <= 0:
    raise RuntimeError("TOP_K chưa hợp lệ. Hãy chạy cell Configuration trước.")
if "SEARCH_PIPELINE_AVAILABLE" not in dir() or not SEARCH_PIPELINE_AVAILABLE:
    raise RuntimeError("search_pipeline is required. Check imports above.")

SEARCH_MODE = "unionWith"
results = run_search(fixture, top_k=TOP_K, mode=SEARCH_MODE)

print("Search mode used:", SEARCH_MODE)
print("Number of results returned:", len(results))


Search mode used: unionWith
Number of results returned: 10


## Step 3 - Results Table

This table shows ranked products with score, price, brand, category, vector/BM25 ranks, channels, matched intent, matched fact, and cold-start signal. Use this section as the main demo output. The explainable fields show why each product matched.


In [9]:
# Bước 3: Hiển thị kết quả với explainable output
# Không dùng pandas để notebook nhẹ và dễ chạy trên mọi máy.

if "results" not in dir():
    raise RuntimeError("results chưa có. Hãy chạy cell Search Execution trước.")

def short_text(value, max_len=60):
    text = "" if value is None else str(value).replace("\n", " ")
    return text if len(text) <= max_len else text[: max_len - 3] + "..."

def format_vnd(value):
    if value in (None, ""):
        return ""
    try:
        return f"{int(value):,} VND"
    except (TypeError, ValueError):
        return str(value)

def channels_for(result):
    channels = (result.get("debug") or {}).get("matched_channels") or []
    if isinstance(channels, list):
        return channels
    return [channels]

print("Rank | Title | Score | Price | Brand | Category | Vector Rank | BM25 Rank | Channels | Matched Intent | Matched Fact | Cold Start")
print("---: | --- | ---: | --- | --- | --- | ---: | ---: | --- | --- | --- | ---")
for rank, result in enumerate(results, start=1):
    debug = result.get("debug") or {}
    print(
        f"{rank} | "
        f"{short_text(result.get('title'))} | "
        f"{result.get('score')} | "
        f"{format_vnd(debug.get('price_vnd'))} | "
        f"{debug.get('brand') or ''} | "
        f"{debug.get('category_id') or ''} | "
        f"{result.get('rank_vector')} | "
        f"{result.get('rank_bm25')} | "
        f"{', '.join(str(channel) for channel in channels_for(result))} | "
        f"{short_text(result.get('matched_intent'))} | "
        f"{short_text(result.get('matched_fact'))} | "
        f"{'yes' if result.get('cold_start_note') else 'no'}"
    )


Rank | Title | Score | Price | Brand | Category | Vector Rank | BM25 Rank | Channels | Matched Intent | Matched Fact | Cold Start
---: | --- | ---: | --- | --- | --- | ---: | ---: | --- | --- | --- | ---
1 | ZHIYIWU Designed for Samsung Galaxy A14 5G Case Clear Sho... | 0.11592527192297686 | 349,750 VND | ZHIYIWU | cell_phones_and_accessories | 1 | 3 | bm25, vector | silicone case for samsung galaxy a14 5g with ring holder | The case is designed for Samsung Galaxy A14 5G 2022. | yes
2 | STENES Sparkle Case Compatible with Samsung Galaxy A14 5G... | 0.11543630952380952 | 398,750 VND | STENES | cell_phones_and_accessories | 3 | 4 | bm25, vector | case for samsung galaxy a14 5g 6.8 inch 2022 | Case is compatible with Samsung Galaxy A14 5G Case 6.8 in... | yes
3 | QIVSTAR Galaxy A14 5G Wallet Case Embossed PU Leather Fli... | 0.11478494623655915 | 274,750 VND | QIVSTAR | cell_phones_and_accessories | 12 | 2 | vector, bm25 | samsung galaxy a14 5g compatible wallet case with card ho... | Com

## Cold Start Analysis

This section filters cold-start results and shows why they appeared. Focus on matched intent and matched fact. This demonstrates how new products can be retrieved without clicks or purchase history.


In [10]:
# Cold Start Analysis — items nào được tìm thấy nhờ HyPE/proposition
# Lọc các result có cold_start_note để giải thích vì sao item cold-start xuất hiện.

if "results" not in dir():
    raise RuntimeError("results chưa có. Hãy chạy cell Search Execution trước.")

cold_results = [result for result in results if result.get("cold_start_note")]
print("Cold-start items in results:", len(cold_results))

if not cold_results:
    print("No cold-start items found in this result set.")
else:
    print("Rank | Item ID | Why it appeared | Matched Intent | Matched Fact")
    print("---: | --- | --- | --- | ---")
    for rank, result in enumerate(cold_results, start=1):
        reason = "HyPE/vector matched buyer intent; BM25/proposition matched factual or keyword evidence."
        print(
            f"{rank} | "
            f"{result.get('item_id')} | "
            f"{reason} | "
            f"{short_text(result.get('matched_intent'), 80)} | "
            f"{short_text(result.get('matched_fact'), 80)}"
        )


Cold-start items in results: 10
Rank | Item ID | Why it appeared | Matched Intent | Matched Fact
---: | --- | --- | --- | ---
1 | B08XWXYJFN | HyPE/vector matched buyer intent; BM25/proposition matched factual or keyword evidence. | silicone case for samsung galaxy a14 5g with ring holder | The case is designed for Samsung Galaxy A14 5G 2022.
2 | B0BNNXH8Q4 | HyPE/vector matched buyer intent; BM25/proposition matched factual or keyword evidence. | case for samsung galaxy a14 5g 6.8 inch 2022 | Case is compatible with Samsung Galaxy A14 5G Case 6.8 inch 2022.
3 | B0BNHVDNTZ | HyPE/vector matched buyer intent; BM25/proposition matched factual or keyword evidence. | samsung galaxy a14 5g compatible wallet case with card holder | Compatible with Samsung Galaxy A14 5G only.
4 | B0BSC8W389 | HyPE/vector matched buyer intent; BM25/proposition matched factual or keyword evidence. | protect samsung galaxy a14 5g with love heart design case | The case is compatible with Samsung Galaxy A14 5G.
5 

## Explainability - Why did item #1 rank high?

This section opens the debug dict for the top result. Review raw scores, ranks, fusion score, bonuses, and matched channels. It makes the ranking auditable instead of black-box.


In [11]:
# Debug view — xem chi tiết scoring cho item top 1
# In full debug dict và các score/rank chính để teammate dễ explain trong video.

if "results" not in dir():
    raise RuntimeError("results chưa có. Hãy chạy cell Search Execution trước.")

if not results:
    print("No results to debug.")
else:
    top_result = results[0]
    debug = top_result.get("debug") or {}
    print("Top item:", top_result.get("item_id"), "-", top_result.get("title"))
    print("raw_vector_score:", debug.get("raw_vector_score"))
    print("raw_bm25_score:", debug.get("raw_bm25_score"))
    print("rank_vector:", top_result.get("rank_vector"))
    print("rank_bm25:", top_result.get("rank_bm25"))
    print("fusion_score:", top_result.get("fusion_score"))
    print("multi_channel_bonus:", debug.get("multi_channel_bonus"))
    print("cold_start_boost:", debug.get("cold_start_boost"))
    print("matched_channels:", debug.get("matched_channels"))
    print("\nFull debug dict")
    print(json.dumps(debug, indent=2, ensure_ascii=False, default=str))


Top item: B08XWXYJFN - ZHIYIWU Designed for Samsung Galaxy A14 5G Case Clear Shockproof Silicone Protective Case with Ring Holder Kickstand Durable Soft TPU Anti-Scratch Non-Yellowing Phone Cover - Clear
raw_vector_score: 0.8630272746086121
raw_bm25_score: 23.74683952331543
rank_vector: 1
rank_bm25: 3
fusion_score: 0.016185271922976842
multi_channel_bonus: 0.05
cold_start_boost: 0.03
matched_channels: ['bm25', 'vector']

Full debug dict
{
  "raw_vector_score": 0.8630272746086121,
  "raw_bm25_score": 23.74683952331543,
  "matched_channels": [
    "bm25",
    "vector"
  ],
  "vector_contribution": 0.009836065573770491,
  "bm25_contribution": 0.006349206349206349,
  "multi_channel_bonus": 0.05,
  "cold_start_boost": 0.03,
  "content_richness_bonus": 0.01974,
  "best_vector": {
    "channel": "vector",
    "matched_intent": "silicone case for samsung galaxy a14 5g with ring holder",
    "matched_fact": null,
    "rank_vector": 1,
    "rank_bm25": null,
    "fusion_score": 0.009836065573770

## Demo Summary

This final section summarizes whether the query was processed, filters were applied, results returned, hybrid channels worked, and cold-start items appeared. Use the final status as the quick readiness check for recording.


In [12]:
# Tổng kết demo thành bảng ngắn để biết pipeline đã sẵn sàng record video chưa.

if "fixture" not in dir() or fixture is None:
    raise RuntimeError("fixture chưa được tạo. Hãy chạy cell Query Processing trước.")
if "results" not in dir():
    raise RuntimeError("results chưa có. Hãy chạy cell Search Execution trước.")
if "channels_for" not in dir():
    raise RuntimeError("channels_for chưa được tạo. Hãy chạy cell Results Display trước.")

query_processed = bool(fixture.get("query_embedding")) and len(fixture.get("query_embedding", [])) == 1024
filters = fixture.get("hard_filters") or {}
filters_applied = bool(filters)
channel_set = set()
for result in results:
    channel_set.update(channels_for(result))
hybrid_label = "vector + bm25" if {"vector", "bm25"}.issubset(channel_set) else "vector only"
cold_count = len([result for result in results if result.get("cold_start_note")])
pipeline_ready = query_processed and len(results) > 0

print("Metric | Value")
print("--- | ---")
print(f"Query processed | {'YES' if query_processed else 'NO'}")
print(f"Language detected | {fixture.get('language_detected')}")
print(f"Hard filters applied | {'YES' if filters_applied else 'NO'} - {json.dumps(filters, ensure_ascii=False)}")
print(f"Results returned | {len(results)} items")
print(f"Hybrid channels | {hybrid_label}")
print(f"Cold start items in top 10 | {cold_count}")
print(f"Pipeline status | {'READY' if pipeline_ready else 'NOT READY'}")


Metric | Value
--- | ---
Query processed | YES
Language detected | vi
Hard filters applied | YES - {"in_stock": true, "price_max": 500000}
Results returned | 10 items
Hybrid channels | vector + bm25
Cold start items in top 10 | 10
Pipeline status | READY
